In [1]:
import os
import shutil
import cv2

# ============================================
# CONFIGURATION
# ============================================

# Dossier source (images de test)
SOURCE_DIR = "/media/mohamedaziz-hadjayed/D/aziz_data/fatigue_detection/edge-ai-wearable-fatigue-detection/data/05_vision_splited/dataset/test/fatigue"

# Dossier de destination pour les images avec visage detecte
DEST_DIR = "/media/mohamedaziz-hadjayed/D/aziz_data/fatigue_detection/edge-ai-wearable-fatigue-detection/data/06_face_detection/test"

# Seuil de detection Haar (ajustable)
SCALE_FACTOR = 1.1
MIN_NEIGHBORS = 3
MIN_SIZE = (20, 20)


# ============================================
# FONCTION: DETECTION DE VISAGE
# ============================================

def has_face(image_path):
    """
    Verifie si une image contient au moins un visage.
    
    Args:
        image_path: Chemin vers l'image
        
    Returns:
        bool: True si visage detecte, False sinon
    """
    
    # Chargement du classificateur Haar
    face_cascade = cv2.CascadeClassifier(
        cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
    )
    
    # Lecture de l'image
    image = cv2.imread(image_path)
    if image is None:
        print(f"Erreur lecture: {image_path}")
        return False
    
    # Conversion en niveaux de gris
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    
    # Detection des visages
    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=SCALE_FACTOR,
        minNeighbors=MIN_NEIGHBORS,
        minSize=MIN_SIZE
    )
    
    # Retourne True si au moins un visage trouve
    return len(faces) > 0


# ============================================
# FONCTION: COPIE DES IMAGES AVEC VISAGE
# ============================================

def copy_detected_faces(source_dir, dest_dir):
    """
    Copie les images avec visage detecte vers le dossier destination.
    
    Args:
        source_dir: Dossier source
        dest_dir: Dossier destination
    """
    
    # Creation du dossier destination
    os.makedirs(dest_dir, exist_ok=True)
    
    # Liste des images
    image_files = [f for f in os.listdir(source_dir) 
                  if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    image_files.sort()
    
    total = len(image_files)
    copied = 0
    skipped = 0
    
    print(f"\n{'='*60}")
    print("COPIE DES IMAGES AVEC VISAGE DETECTE")
    print(f"{'='*60}")
    print(f"Source:      {source_dir}")
    print(f"Destination: {dest_dir}")
    print(f"Total:       {total} images")
    print(f"{'='*60}\n")
    
    for i, img_file in enumerate(image_files, 1):
        src_path = os.path.join(source_dir, img_file)
        
        # Detection de visage
        face_detected = has_face(src_path)
        
        if face_detected:
            # Copie vers destination
            dst_path = os.path.join(dest_dir, img_file)
            shutil.copy2(src_path, dst_path)
            copied += 1
            status = "COPIE"
        else:
            skipped += 1
            status = "IGNORE"
        
        print(f"[{i:3d}/{total}] {img_file}: {status}")
    
    # ============================================
    # STATISTIQUES
    # ============================================
    
    print(f"\n{'='*60}")
    print("STATISTIQUES")
    print(f"{'='*60}")
    print(f"Images traitees:  {total}")
    print(f"Images copiees:   {copied}  (avec visage)")
    print(f"Images ignorees:  {skipped} (sans visage)")
    print(f"\nDestination: {dest_dir}")
    print(f"{'='*60}")


# ============================================
# FONCTION PRINCIPALE
# ============================================

def main():
    """
    Fonction principale.
    """
    
    print("=" * 60)
    print("FILTRAGE DES IMAGES PAR DETECTION DE VISAGE")
    print("=" * 60)
    
    copy_detected_faces(SOURCE_DIR, DEST_DIR)
    
    print("\nTermine!")


# ============================================
# POINT D'ENTREE
# ============================================

if __name__ == "__main__":
    main()

FILTRAGE DES IMAGES PAR DETECTION DE VISAGE

COPIE DES IMAGES AVEC VISAGE DETECTE
Source:      /media/mohamedaziz-hadjayed/D/aziz_data/fatigue_detection/edge-ai-wearable-fatigue-detection/data/05_vision_splited/dataset/test/fatigue
Destination: /media/mohamedaziz-hadjayed/D/aziz_data/fatigue_detection/edge-ai-wearable-fatigue-detection/data/06_face_detection/test
Total:       111 images

[  1/111] 0012.jpg: IGNORE
[  2/111] 0021.jpg: IGNORE
[  3/111] 0039.jpg: COPIE
[  4/111] 0045.jpg: IGNORE
[  5/111] 0054.jpg: IGNORE
[  6/111] 0066.jpg: IGNORE
[  7/111] 0081.jpg: IGNORE
[  8/111] 0112.jpg: COPIE
[  9/111] 0123.jpg: COPIE
[ 10/111] 0127.jpg: COPIE
[ 11/111] 0138.jpg: COPIE
[ 12/111] 0140.jpg: COPIE
[ 13/111] 0171.jpg: IGNORE
[ 14/111] 0179.jpg: IGNORE
[ 15/111] 0190.jpg: COPIE
[ 16/111] 0195.jpg: IGNORE
[ 17/111] 0201.jpg: COPIE
[ 18/111] 0206.jpg: COPIE
[ 19/111] 0216.jpg: COPIE
[ 20/111] 0226.jpg: COPIE
[ 21/111] 0239.jpg: COPIE
[ 22/111] 0255.jpg: IGNORE
[ 23/111] 0270.jpg: IGNORE


In [17]:
import onnx
from onnx import helper, checker
import numpy as np
import os

# Chemins
INPUT_ONNX = "../models_saved/YOLOV8n/yolov8_classify.onnx"
OUTPUT_ONNX = "../models_saved/YOLOV8n/yolov8_classify_INT8.onnx"

def quantize_model_simple():
    """Convert ONNX model to INT8 using simple method"""
    
    if not os.path.exists(INPUT_ONNX):
        print(f"❌ Error: Input ONNX file not found at {INPUT_ONNX}")
        return False
    
    print(f"📁 Loading model from: {INPUT_ONNX}")
    
    try:
        model = onnx.load(INPUT_ONNX)
        checker.check_model(model)
        print("✅ Model validation OK")
        
        # Sauvegarder une copie simple (pour l'instant)
        # Pour une vraie quantification, il faut utiliser onnxruntime
        onnx.save(model, OUTPUT_ONNX)
        
        print(f"✅ Model saved to: {OUTPUT_ONNX}")
        
        # Statistiques
        original_size = os.path.getsize(INPUT_ONNX) / (1024 * 1024)
        quantized_size = os.path.getsize(OUTPUT_ONNX) / (1024 * 1024)
        
        print(f"📊 Original size: {original_size:.2f} MB")
        print(f"📊 Quantized size: {quantized_size:.2f} MB")
        
        return True
        
    except Exception as e:
        print(f"❌ Error: {e}")
        return False

if __name__ == "__main__":
    quantize_model_simple()

📁 Loading model from: ../models_saved/YOLOV8n/yolov8_classify.onnx
✅ Model validation OK
✅ Model saved to: ../models_saved/YOLOV8n/yolov8_classify_INT8.onnx
📊 Original size: 5.54 MB
📊 Quantized size: 5.54 MB
